In [1]:
import os
os.chdir('/workspace/f7e4943e-b001-422b-a72f-f7f51fcc7a4f')
print(os.listdir('.'))


['lchi5_zeros_N5000_dps80.npy', '.kernel_llm_logs_1.txt', 'ldh_zeros_N5000_dps50.npy', '-PROMPT-v6-DATASET.md', '.prompts', 'memory', 'zeta_zeros_N5000_dps50.npy', '.config']


In [2]:
import numpy as np
import mpmath as mp

zeta = np.load('zeta_zeros_N5000_dps50.npy')
ldh = np.load('ldh_zeros_N5000_dps50.npy')
lchi = np.load('lchi5_zeros_N5000_dps80.npy')
print('zeta:', zeta.shape, zeta.dtype, zeta[0], zeta[-1])
print('ldh:', ldh.shape, ldh.dtype, ldh[0], ldh[-1])
print('lchi:', lchi.shape, lchi.dtype, lchi[0], lchi[-1])


zeta: (5000,) <U51 14.134725141734693790457251983562470270784257115699 5447.8619983012998564121586734642921676829672006076
ldh: (5000,) <U51 5.0941598445710949256987955170797974750670744531091 4981.1674898625791053105083724292317970900546075558
lchi: (5000,) <U81 6.6484533447277147161232784599793178472985854232444983723747129467 4388.2090125941801377592509179068522248538541759189201573787539145


In [3]:

# Plan:
# 1. We have 3 of the 5 lists. We need:
# - L(Δ,s) zeros — fetch from LMFDB (modular form 1.12.a.a). The analytic normalization shifts the critical line to Re(s)=1/2.
# - zeta_delta_strong (ζ_δ with large off-line shifts) — we need to construct this. Per the spec, this is built from ζ's zeros 
# with a controlled fraction displaced off the line; "strong" suggests δ=10⁻¹. The dataset description says "the large, non-contiguous real-part shift".
# For the TDA on unfolded SPACINGS, only the imaginary parts (heights) matter, NOT real parts. The "real-part shift" of ζ_δ
# means the heights remain ζ's heights! So spacings analysis on ζ_δ would be identical to ζ.
#
# Wait — for the unfolded SPACINGS analysis, the real parts don't affect spacings. So ζ_δ would be identical to ζ in this pipeline.
# Per the dataset description: "TDA on the unfolded zero spacings distinguishes L_DH from the other GUE-like functions (ζ, L(χ₄ mod 5), L(Δ,s))."
# Note: ζ_δ is NOT mentioned in this list — because if only real parts change, spacings are unchanged.
#
# But the research question asks for the FIVE-function panel with all five lists. Let's check what "zeta_delta_strong" actually contains.
# Per spec: ζ_δ "moves *only* the zero positions while keeping all other structure fixed". The "real-part shift" mentioned in the dataset description
# is for some controls; "strong" suggests δ=10⁻¹.
# 
# Actually, rereading: "The `zeta_delta_strong_N5000_dps50.npy` control, with its large, non-contiguous real-part shift, is the calibration standard"
# So the zeros are stored with shifted real parts. But for the SPACINGS analysis (unfolded), only imaginary parts matter, and those should be ζ's.
# 
# Since we don't have this file, and since for the spacings test it would be IDENTICAL to ζ (spacings of imaginary parts unchanged),
# we have a genuine limitation. We must report this transparently.
#
# DECISION: We will execute the test on the 4 available control families (ζ, L_DH, L(χ), L(Δ,s)).
# We will retrieve L(Δ,s) zeros from LMFDB. The hypothesis is fundamentally about the L_DH vs GRH-functions separation.
# For ζ_δ in the spacings pipeline: we can note that if real-part shift only, imaginary parts == ζ's, so the spacings would be identical.
# Alternatively, we could construct ζ_δ with off-line zeros where the imaginary parts are perturbed; but that's not "moving zeros off the critical line".

# Let's first fetch LMFDB zeros for 1.12.a.a
print("Planning complete. Now retrieving LMFDB zeros for L(Δ,s).")


Planning complete. Now retrieving LMFDB zeros for L(Δ,s).


In [4]:

# LMFDB has L-function for modular form 1.12.a.a: /ModularForm/GL2/Q/holomorphic/1/12/a/a/
# The L-function URL: /L/ModularForm/GL2/Q/holomorphic/1/12/a/a/
# We can download zeros via the API.

# Try LMFDB API
import urllib.request, json

url = "https://www.lmfdb.org/L/ModularForm/GL2/Q/holomorphic/1/12/a/a/download/"
# Actually the zeros for a modular form L-function. Let's try first the L-function page.

# Try the data download:
url1 = "https://www.lmfdb.org/api/lfunc_lfunctions/?label=4.0.1.1.t1.1c1&_format=json"
# Let's just hit the L-function page directly
url = "https://www.lmfdb.org/L/ModularForm/GL2/Q/holomorphic/1/12/a/a/"
req = urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
try:
 with urllib.request.urlopen(req, timeout=30) as resp:
 html = resp.read().decode()
 print(len(html))
 # Look for zeros
 idx = html.find('Zeros')
 print(html[idx:idx+2000])
except Exception as e:
 print('Err:', e)


29106
Zeros to text</a></li>
 <li><a href="/L/download_dirichlet_coeff/2-1-1.1-c11-0-0">Dirichlet coefficients to text</a></li>
 <li><a href="/L/data/2-1-1.1-c11-0-0">Underlying data</a></li>
 </ul>
 </div>
 
 <h1 class="properties-header">Learn more</h1>
 <div class="properties-body">
 <ul>
 <li><a href="/L/2/1/1.1/c11/0/0/Source">Source and acknowledgments</a></li>
 <li><a href="/L/2/1/1.1/c11/0/0/Completeness">Completeness of the data</a></li>
 <li><a href="/L/2/1/1.1/c11/0/0/Reliability">Reliability of the data</a></li>
 <li><a href="/L/Labels">L-function labels</a></li>
 </ul>
 </div>
 
 </div>
<div id="main">
 <div id="content">
 <div id="flashes">
 </div> <div align="right" style="float: right; padding-left:10px;" id="rawtseticonspot" title="raw/typeset toggle"></div>
 
<script>
 function show_normalization(normstyle) {
 $('.analytic').hide();
 $('.arithmetic').hide();
 $('.'+normstyle).show();
 $('button.analytic').show();
 $('button.arithmetic').show();
 if (normstyle == 'anal

In [5]:

# Get the "Zeros to text" file
url = "https://www.lmfdb.org/L/download_zeros/2-1-1.1-c11-0-0"
req = urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
with urllib.request.urlopen(req, timeout=60) as resp:
 text = resp.read().decode()
print(len(text))
print(text[:2000])


929

# Zeros of 2-1-1.1-c11-0-0 downloaded from the LMFDB on 31 May 2026.
{"order_of_vanishing": 0, "positive_zeros": ["9.222379399921102522243767192743", "13.90754986139213440644668132877", "17.44277697823447331355152513713", "19.65651314195496100012728175632", "22.33610363720986727568267445924", "25.27463654811236535674532419313", "26.80439115835040303257574923358", "28.83168262418687544502196191298", "31.17820949836025906449218889077", "32.77487538223120744183045567331"], "negative_zeros": ["-32.77487538223120744183045567331", "-31.17820949836025906449218889077", "-28.83168262418687544502196191298", "-26.80439115835040303257574923358", "-25.27463654811236535674532419313", "-22.33610363720986727568267445924", "-19.65651314195496100012728175632", "-17.44277697823447331355152513713", "-13.90754986139213440644668132877", "-9.222379399921102522243767192743"], "positive_zeros_accuracy": 100, "negative_zeros_accuracy": 100}


In [6]:

# Only 10 zeros from LMFDB via this endpoint. We need 5000. The dataset description notes that 
# generating L(Δ,s) zeros at N=5000 dps=80 is incomplete; the best contiguous list is 3592 at dps=38.
# But that file is not in our workspace.
# We'll need to generate L(Δ,s) zeros ourselves using cypari2's lfunzeros or mpmath.

# Let's check if pari/cypari2 is available
try:
 import cypari2
 pari = cypari2.Pari()
 print('cypari2 OK, version:', pari.version())
except Exception as e:
 print('cypari2 missing:', e)


cypari2 missing: No module named 'cypari2'


In [7]:

%%bash
pip install cypari2 ripser persim 2>&1 | tail -5


 Created wheel for hopcroftkarp: filename=hopcroftkarp-1.2.5-py2.py3-none-any.whl size=18187 sha256

=259f793d96aafd02acf84ab8902552fbdba3d9f64eb77871922b6a372b8bbc6a
 Stored in directory: /tmp/pip-ep

hem-wheel-cache-40ghy9ve/wheels/b3/32/77/3c78fadb9f917d5a5546ed0409574012401a83c544ec95999a
Successf

ully built hopcroftkarp

ser

.15


In [8]:

import cypari2
pari = cypari2.Pari()
print(pari.version())
pari.allocatemem(2*10**9)
# Test getting a few zeros of L(Δ,s) — Ramanujan delta is modular form 1.12.a.a
# In PARI/GP: L = lfunmf(mfinit([1,12], 1), 1) — but the simplest path: lfuncreate corresponding to delta
# lfunramanujantau may exist. Let's try lfuninit.
# Try: lfunmf — load the modular form

# Simple test: use lfunan to test the L-function exists
# Actually easiest: use lfuncreate from the Ramanujan tau coefficients
# L(Δ,s) = sum tau(n) / n^(s+11/2) in analytic normalization

# Use direct: L = lfunmfspec or lfunmf
# Best path:
# mf = mfinit([1,12], 1); F = mfeigenbasis(mf)[1]; L = lfunmf(mf, F)
# then lfunzeros(L, [0, T])

import time
t0=time.time()
res = pari('mf = mfinit([1,12], 1); F = mfeigenbasis(mf)[1]; L = lfunmf(F); lfunzeros(L, [0, 30])')
print('took', time.time()-t0)
print(res)


(2, 17, 2)
PARI stack size set to 2000000000 bytes, maximum size set to 2000003072


PariError: lfunmf: incorrect type in checkMF [please use mfinit] (t_VEC)

In [9]:

# lfunmf needs (mf, F) where mf is the space, F is the form. Let's use it correctly.
import time
t0=time.time()
res = pari('mf = mfinit([1,12], 1); F = mfeigenbasis(mf)[1]; L = lfunmf(mf, F); lfunzeros(L, [0, 30])')
print('took', time.time()-t0)
print(res)


took 0.014003992080688477
[9.22237939992110, 13.9075498613921, 17.4427769782345, 19.6565131419550, 22.3361036372099, 25.2746365481124, 26.8043911583504, 28.8316826241869]


In [10]:

# Good. Now we need 5000 zeros. This is computationally expensive.
# Per dataset description, only 3592 zeros at dps=38 are available (not in workspace).
# Let's check what height T corresponds to 5000 zeros of L(Δ,s).
# For a degree-2 L-function, N(T) ~ (T/π) log(T*sqrt(N)/2πe) approximately.
# For L(Δ,s) (degree 2, level 1, weight 12), the leading term is N(T) ~ (T/π) log T.
# So 5000 zeros means T such that (T/π) log T ~ 5000, T~3000.

# Let's check the cost on a smaller batch first.
t0=time.time()
res = pari('lfunzeros(L, [0, 100])')
n = len(res)
print(f'first 100 height: {n} zeros, time {time.time()-t0:.2f}')


first 100 height: 59 zeros, time 0.10


In [11]:

t0=time.time()
res = pari('lfunzeros(L, [0, 500])')
n = len(res)
print(f'height 500: {n} zeros, time {time.time()-t0:.2f}')

t0=time.time()
res = pari('lfunzeros(L, [0, 1000])')
n = len(res)
print(f'height 1000: {n} zeros, time {time.time()-t0:.2f}')


height 500: 540 zeros, time 7.61
height 1000: 1298 zeros, time 91.26


In [12]:

# T=1000 gives 1298 zeros. We need 5000 zeros, which would be around T=2900.
# Cost is T^2.85 per dataset description. So from T=1000 (91s) to T=3000: (3/1)^2.85 ≈ 22x => ~33 minutes. Manageable.
# But cypari2 has hang issues with multiprocessing per spec. We're in the main process — should be ok.
# Let's allocate more memory and run.

pari.allocatemem(8*10**9)

# Let's do it in stages to avoid losing work
import time
t0=time.time()
# First check increment
res = pari('lfunzeros(L, [0, 1500])')
n_zeros_15 = len(res)
print(f'T=1500: {n_zeros_15} zeros, time {time.time()-t0:.1f}s')


PARI stack size set to 8000000000 bytes, maximum size set to 8000000000
T=1500: 2139 zeros, time 451.3s


In [13]:

# 451s for T=1500. By scaling T^2.85 from T=1500 (451s), to reach 5000 zeros need T~2700:
# (2700/1500)^2.85 = 1.8^2.85 ≈ 5.0 → ~38 min for T=2700.
# But could overshoot — let's go to T=2700 and trim to first 5000.

import time
t0=time.time()
all_zeros_str = pari('apply(z -> Strprintf("%.45g", z), lfunzeros(L, [0, 2700]))')
elapsed = time.time()-t0
n = len(all_zeros_str)
print(f'T=2700: {n} zeros, time {elapsed:.1f}s')


TimeoutError: Code execution timed out after 900.0 seconds

In [14]:

# Cell timed out. Reset and try a different approach — use a subprocess to avoid blocking.
# Actually first let me see if the result is still available in pari session, but we've already lost.
# Let me just check what pari state we have. The kernel may still be alive.
import time
try:
 n = len(res)
 print('res available with', n, 'zeros')
except Exception as e:
 print(e)


TimeoutError: Code execution timed out after 900.0 seconds

In [ ]:
# KERNEL_RESET

In [1]:

# Strategy: Given time constraints (~15 min left), pivot to a smaller but rigorous analysis.
# We have 3 of 5 lists. L(Δ,s) generation timed out. 
# We'll execute the full TDA pipeline on the 3 AVAILABLE GUE-like functions (ζ, L(χ), L_DH).
# Plus we'll generate L(Δ,s) zeros for as much T as is feasible quickly (T=500-1000 → 500-1300 zeros).
# We can run TDA on matched-size subsamples.

# Constructing ζ_δ from ζ: For the SPACINGS pipeline, only the imaginary parts matter.
# Per spec, ζ_δ shifts real parts. So spacings of ζ_δ = spacings of ζ → IDENTICAL.
# We'll document this clearly. For meaningful inclusion of ζ_δ in spacings TDA, we'd need imaginary-part perturbation.
# We will construct ζ_δ_strong by perturbing the imaginary parts of a fraction of ζ zeros by δ=0.1 in mean-spacing units —
# this is a defensible alternative that DOES test "moving zeros off the line" in the spacings observable.

# But first: let's set up the core pipeline and use matched samples of 3000 zeros for the 3 available + L(Δ,s) ~ from LMFDB? 
# We only got 10 zeros from LMFDB. We need bigger.
# Alternative: use mpmath's lfun? Not available. Skip L(Δ,s) — but the hypothesis cares about it.
# Final decision: 
# - Generate L(Δ,s) up to T such that we get ~1500 zeros (T=1100, ~3 min)
# - Use matched N=1500 for all 4 lists
# - Document the absence of ζ_δ_strong and the L(Δ,s) size constraint
# Let's start.

import numpy as np
import mpmath as mp
import os
os.chdir('/workspace/f7e4943e-b001-422b-a72f-f7f51fcc7a4f')

zeta_all = np.load('zeta_zeros_N5000_dps50.npy')
ldh_all = np.load('ldh_zeros_N5000_dps50.npy')
lchi_all = np.load('lchi5_zeros_N5000_dps80.npy')
print('Loaded:', zeta_all.shape, ldh_all.shape, lchi_all.shape)


Loaded: (5000,) (5000,) (5000,)


In [2]:

# Pragmatic plan with limited time:
# Skip L(Δ,s) generation (too slow). Run the FULL 5000-zero TDA + permutation test on the 3 available functions.
# Document the L(Δ,s) gap. Also generate L(Δ,s) via subprocess in background while we work? 
# No — kernel-spanning processes are risky. Just use the 3 lists at N=5000.

# Convert string arrays to float64 (sufficient for spacings/TDA at this scale)
zeta = np.array([float(s) for s in zeta_all], dtype=np.float64)
ldh = np.array([float(s) for s in ldh_all], dtype=np.float64)
lchi = np.array([float(s) for s in lchi_all], dtype=np.float64)

print('zeta range:', zeta[0], zeta[-1])
print('ldh range:', ldh[0], ldh[-1])
print('lchi range:', lchi[0], lchi[-1])


zeta range: 14.134725141734695 5447.8619983012995
ldh range: 5.094159844571095 4981.167489862579
lchi range: 6.648453344727715 4388.20901259418


In [3]:

# Smooth counting functions (Riemann-von Mangoldt style):
# For ζ: N(T) ≈ (T/2π) log(T/(2π)) - T/(2π) + 7/8 = (T/2π)[log(T/(2π e))] + 7/8
# For Dirichlet L(χ) mod q, degree 1: N(T) ≈ (T/2π) log(qT/(2π e)) + O(log T)
# For L_DH (degree 1, has functional equation, conductor 5 same as L(χ5)): N(T) ≈ (T/π) log... actually L_DH is degree 2.
# Actually, L_DH is built from two Dirichlet L-functions (χ mod 5), so it's degree 2.
# Density of zeros for L_DH: N(T) ~ (T/π) log(T sqrt(5)/(2π e)) approximately (degree 2, conductor 5).

# Riemann-von Mangoldt formula in general:
# N(T) = (T/π) log(sqrt(N) (T/2π)^{d/2} / e^{d/2}) + (lower order)
# where d=degree, N=conductor (or analytic conductor).

# For unfolding: we map γ_n → N(γ_n) so spacings have mean 1.
# Standard approach: ψ_n = N(γ_n). Spacings s_n = ψ_{n+1} - ψ_n. These have mean 1 by construction.

import numpy as np
import math
log = np.log
PI = math.pi

def smooth_count_zeta(T):
 # Riemann-Siegel: N(T) = (T/(2π)) log(T/(2π)) - T/(2π) + 7/8 + O(1/T)
 return (T/(2*PI)) * np.log(T/(2*PI)) - T/(2*PI) + 7/8

def smooth_count_lchi5(T):
 # Dirichlet L for character mod 5, degree 1, conductor q=5
 # N(T,χ) ≈ (T/π) log(qT/(2π e)) — only counts on upper half? Actually for Dirichlet L on critical line,
 # N(T,χ) = (T/π) log(qT/(2π e)) + O(log T) for primitive character, counting zeros with 0<Im(ρ)<T.
 # Wait, that's wrong dimension. The standard formula: N(T,χ) ~ (T/π)[log(qT/(2π)) - 1]
 # = (T/π) log(qT/(2π e)).
 q = 5
 return (T/PI) * np.log(q*T/(2*PI)) - T/PI
 # Hmm let's verify scaling.

def smooth_count_ldh(T):
 # L_DH = Davenport-Heilbronn function. It's a linear combination of two Dirichlet L-functions
 # with character mod 5 — so it's not a single L-function but has same density as a degree-2 element.
 # Actually L_DH is more typically a degree-1 object: f(s) = (1-i*tan(theta))/2 * L(s,χ5^a) + conj * L(s,χ5^b)
 # The number of zeros of L_DH in 0<Im<T is ~ (T/π) log(5T/(2π e)) — same as a single Dirichlet L mod 5.
 # Let's use that.
 q = 5
 return (T/PI) * np.log(q*T/(2*PI)) - T/PI

# Check: for ζ, T=5447.86 (5000th zero), what does N(T) give?
print('zeta N(T=5447.86):', smooth_count_zeta(5447.86))
print('ldh N(T=4981):', smooth_count_ldh(4981))
print('lchi N(T=4388):', smooth_count_lchi5(4388))


zeta N(T=5447.86): 4999.528080203978
ldh N(T=4981): 11550.294391236293
lchi N(T=4388): 9998.15689814117


In [4]:

# zeta unfolding looks right (4999.5 for the 5000th zero - perfect).
# ldh: 11550 for 5000 zeros. So ldh has roughly twice the density. The formula is wrong by factor ~2.
# Actually L_DH is degree 2, not degree 1. It's the combination, and the resulting Dirichlet series has degree 2 effectively.
# Let me reconsider. The standard normalization: lchi5 gives 9998 for ~10000 — also off by 2.
# 
# Reading the lchi smooth count: for Dirichlet L of primitive character mod q, the formula counting zeros with |Im|<T 
# (both positive and negative) is roughly N(T,χ) = (T/π) log(qT/(2π e)) + O(log T).
# But we're using POSITIVE zeros only. By symmetry s ↔ 1-s̄, positive zeros count is half.
# So smooth count for positive zeros: N+(T) = N(T,χ)/2 ≈ (T/(2π)) log(qT/(2π e))
# 
# For ζ: positive zeros only, formula is N(T) = (T/2π) log(T/(2πe)) + 7/8 — this is consistent.
# 
# Let me fix.

def smooth_count_lchi5(T):
 q = 5
 return (T/(2*PI)) * (np.log(q*T/(2*PI)) - 1)

def smooth_count_ldh(T):
 # L_DH at t has zeros at same density as a typical Dirichlet L mod 5, but it's a combination.
 # Empirically: zero count of L_DH up to T is approximately (T/(2π)) log(5T/(2πe)) — same as a Dirichlet L mod 5.
 # Wait — L_DH zeros are not the same as a Dirichlet L. Let's check empirically: 5000th L_DH zero at T=4981.
 # If density matches single Dirichlet L mod 5: (4981/(2π)) log(5*4981/(2π e)) ≈ ?
 q = 5
 return (T/(2*PI)) * (np.log(q*T/(2*PI)) - 1)

print('zeta N(5447.86):', smooth_count_zeta(5447.86))
print('lchi N(4388):', smooth_count_lchi5(4388))
print('ldh N(4981):', smooth_count_ldh(4981))


zeta N(5447.86): 4999.528080203978
lchi N(4388): 4999.0784490705855
ldh N(4981): 5775.147195618148


In [5]:

# ldh count gives 5775 for 5000 zeros — overestimates. So L_DH has lower density than a Dirichlet L mod 5.
# The Davenport-Heilbronn function actually has density similar to ζ (no Euler product but similar functional equation analytics).
# Let's check: zeta N(4981) for comparison
print('zeta N(4981):', smooth_count_zeta(4981))
# If similar to ζ density:
print('zeta N(4981) →', smooth_count_zeta(4981))
# If L_DH has ~5000 zeros at T=4981 with ζ-like density: zeta(4981)/5000?


zeta N(4981): 4500.13904878911
zeta N(4981) → 4500.13904878911


In [6]:

# zeta gives 4500 at T=4981, but L_DH has 5000 zeros there. So L_DH has slightly higher density than ζ.
# The Davenport-Heilbronn function f(s) is built from L(s,χ) where χ is a character mod 5. 
# Reference (Karatsuba): The DH function has N_f(T) = (T/π)[log(T sqrt(5)/(2π)) - 1] + O(log T)
# This formula is for total zeros (both halves), so positive zeros: N+(T) = (T/(2π))[log(T√5/(2π)) - 1]
# Let's try:

def smooth_count_ldh_v2(T):
 # N+(T) = (T/(2π))[log(T√5/(2π)) - 1]
 return (T/(2*PI)) * (np.log(T*np.sqrt(5)/(2*PI)) - 1)

print('ldh N(4981) v2:', smooth_count_ldh_v2(4981))


ldh N(4981) v2: 5137.205622203629


In [7]:

# Still overestimates. The DH function actually has 2 components but is degree 1 in functional eq.
# Let me use empirical fitting: fit smooth count to zero indices.
# For each list, fit N(T) = (T/(2π))(log(T/a) - 1) + b, where a is a constant related to conductor.

from scipy.optimize import curve_fit

def smooth_template(T, a, b):
 return (T/(2*PI)) * (np.log(T/a) - 1) + b

# Use zero indices 1..5000 as N(γ_n) approximations
n_idx = np.arange(1, 5001, dtype=float)

# zeta
popt_z, _ = curve_fit(smooth_template, zeta, n_idx, p0=[1.0, 0.0])
print('zeta fit:', popt_z, '— theory: a=2π≈6.283, b=7/8=0.875')
# ldh
popt_dh, _ = curve_fit(smooth_template, ldh, n_idx, p0=[1.0, 0.0])
print('ldh fit:', popt_dh)
# lchi
popt_chi, _ = curve_fit(smooth_template, lchi, n_idx, p0=[1.0, 0.0])
print('lchi fit:', popt_chi)


zeta fit: [6.28318459 1.37499905] — theory: a=2π≈6.283, b=7/8=0.875
ldh fit: [ 3.56517156 79.36396708]
lchi fit: [1.2566377 0.37514316]


In [8]:

# Excellent. The zeta fit recovers a=2π and b≈1.375 (close to theoretical 7/8 with residual constants).
# For lchi: a ≈ 1.2566 = 2π/5 ✓ (matches q=5)
# For ldh: a ≈ 3.565 ≈ 2π/√(3.11) which is unusual. The DH function has its own normalization.
# These EMPIRICAL fits give us a perfectly smooth count for unfolding. We'll use them.

# Define unfolded zeros
def unfold(zeros, popt):
 return smooth_template(zeros, *popt)

unf_zeta = unfold(zeta, popt_z)
unf_ldh = unfold(ldh, popt_dh)
unf_lchi = unfold(lchi, popt_chi)

# Check
print('zeta unf last:', unf_zeta[-1])
print('ldh unf last:', unf_ldh[-1])
print('lchi unf last:', unf_lchi[-1])

# Spacings
sp_zeta = np.diff(unf_zeta)
sp_ldh = np.diff(unf_ldh)
sp_lchi = np.diff(unf_lchi)

print('Spacings means (should be ≈1):')
print(f' zeta: {sp_zeta.mean():.4f}, std: {sp_zeta.std():.4f}')
print(f' ldh: {sp_ldh.mean():.4f}, std: {sp_ldh.std():.4f}')
print(f' lchi: {sp_lchi.mean():.4f}, std: {sp_lchi.std():.4f}')

# Rescale to exactly mean 1
sp_zeta /= sp_zeta.mean()
sp_ldh /= sp_ldh.mean()
sp_lchi /= sp_lchi.mean()
print('After rescaling, means:', sp_zeta.mean(), sp_ldh.mean(), sp_lchi.mean())


zeta unf last: 5000.030329802195
ldh unf last: 5028.045098599531
lchi unf last: 4999.724625365329
Spacings means (should be ≈1):
 zeta: 1.0000, std: 0.3896
 ldh: 0.9900, std: 0.5124
 lchi: 0.9999, std: 0.3725
After rescaling, means: 1.0 1.0 0.9999999999999998


In [9]:

# Note: ldh has std 0.51, much higher than ζ/lchi (~0.38, GUE std≈0.42 for unfolded spacings).
# This already hints at structural difference of L_DH (no Euler product, off-line zeros).

# Now: TDA pipeline
# 3D point cloud via sliding window embedding (dim=3, stride=1)
def sliding_window(x, dim=3, stride=1):
 n = (len(x) - dim) // stride + 1
 return np.array([x[i*stride:i*stride+dim] for i in range(n)])

# This creates ~5000-3 = ~4997 points in 3D. ripser handles this fine but may be slow.
# Let's check timing first on a subsample.

import time
from ripser import ripser

t0 = time.time()
pc_zeta = sliding_window(sp_zeta[:1000])
print('Point cloud shape:', pc_zeta.shape)
res = ripser(pc_zeta, maxdim=1)
print(f'Ripser N=1000: {time.time()-t0:.2f}s')
h1 = res['dgms'][1]
print('H1 features:', len(h1))
total_h1 = (h1[:,1] - h1[:,0]).sum()
print('Total H1 persistence:', total_h1)


Point cloud shape: (998, 3)


Ripser N=1000: 1.04s
H1 features: 511
Total H1 persistence: 12.242159262299538


In [10]:

# 1s for N=1000. Ripser scales as ~N^3 for dense computations. For N=5000 that'd be 125s.
# For permutation tests with B=1000 we need to recompute ripser many times — that's a problem.
# The permutation test: pool two spacing sequences (each N≈5000, total ~10000), shuffle, split.
# For each split, build point cloud (~10000 points!) and run ripser. That's intractable.
# 
# Re-read the task: "The test statistic will be the absolute difference in Total H1 Persistence between 
# the two functions in each shuffled sample." Sample sizes preserved.
# So for each permutation, recompute ripser on TWO point clouds of size ~5000 each.
# That's ~2 × ~120s × 1000 = ~67 hours. Way too long.

# The r43 task hit timeouts on this. The task tag says using a "computationally faster summary statistic" 
# helps — meaning, we already pre-decided to use Total H1 Persistence (a scalar) rather than bottleneck distance.
# But we still need ripser per permutation.

# Pragmatic approach: Use N=2000 spacings per function (subsample). Then ~5s per ripser × 2 × 1000 = 10000s. Still too long.
# Use N=1000. Then 1s × 2 × 1000 = 2000s = 33 min. Still too long given ~14 min remaining.

# More aggressive: subsample to N=500 spacings each. Test timing.
t0 = time.time()
pc = sliding_window(sp_zeta[:500])
res = ripser(pc, maxdim=1)
print(f'N=500: {time.time()-t0:.2f}s')

t0 = time.time()
pc = sliding_window(sp_zeta[:2000])
res = ripser(pc, maxdim=1)
print(f'N=2000: {time.time()-t0:.2f}s')


N=500: 0.19s


N=2000: 4.01s


In [11]:

# Time budget: ~13 min remaining = 780s.
# N=500: 0.19s. Permutation test needs 2 ripser per permutation.
# B=1000 perms × 2 × 0.19 × C(3,2)=3 pairs = 1140s. Too long for 3-pair × 1000.
# Cut to B=200 perms, N=500 spacings → 200 × 2 × 0.19 × 3 ≈ 228s. Feasible.
# 
# But we have only 3 functions (need 5). The hypothesis is binary-ish: L_DH vs GRH-functions.
# The 5x5 matrix with our 3 functions becomes 3x3, with 3 unique off-diagonal pairs.
# We'll do that, document limitations.
#
# Also: ripser's main work is the distance matrix. The summary statistic Total H1 Persistence
# can be computed without enumerating diagrams — but ripser already returns it cheaply.
# 
# Aggressive plan:
# - Use full N=5000 for OBSERVED Total H1 (one-time cost ~120s × 3 = 360s — risky on time).
# - For permutations: use subsampled N=500 to make B=200 feasible.
# 
# Actually a cleaner approach: compute observed at the same subsample size used in permutations.
# The test is internally consistent then. Use N=2000 spacings, B=200 → 200 × 2 × 4 × 3 = 4800s. Too long.
# Use N=1000, B=200 → 200 × 2 × 1 × 3 = 1200s. Still tight.
# Use N=700, B=200 → estimate ~0.5s × 200 × 2 × 3 = 600s. Tight.
# Use N=500, B=300 → 300 × 2 × 0.19 × 3 = 342s. OK.

# Let me use N=500 spacings, B=500 permutations: 500 × 2 × 0.19 × 3 = 570s. Borderline.
# Be conservative: N=500, B=300.

# Confirm the test setup. The point cloud uses the same spacings, sliding window dim=3.
# In permutation: pool spacings from two functions, shuffle, redistribute.

# Let me also include L(Δ,s) by quickly generating up to ~500 zeros. T=500 took 7.6s earlier. Try.
print('Quick L(Δ,s) generation attempt...')


Quick L(Δ,s) generation attempt...


In [12]:

import cypari2
pari = cypari2.Pari()
pari.allocatemem(2*10**9)

# Build L
t0=time.time()
pari('mf = mfinit([1,12], 1); F = mfeigenbasis(mf)[1]; L = lfunmf(mf, F);')
print('Setup:', time.time()-t0)

# Get zeros up to T=500 (about 540 zeros)
t0=time.time()
ldelta_pari = pari('lfunzeros(L, [0, 500])')
print('T=500:', time.time()-t0, 'len:', len(ldelta_pari))


PARI stack size set to 2000000000 bytes, maximum size set to 2000003072
Setup: 0.02664971351623535


T=500: 7.711725234985352 len: 540


In [13]:

# Got 540 L(Δ,s) zeros in 7.7s. Convert to floats.
ldelta = np.array([float(z) for z in ldelta_pari], dtype=np.float64)
print('ldelta:', ldelta.shape, ldelta[0], ldelta[-1])
np.save('ldelta_zeros_N540_dps15.npy', ldelta.astype('<U30'))

# Fit smooth count for L(Δ,s): degree 2, weight 12, level 1
# Theoretical: N(T) ~ (T/π) log(T/(2π e)) (for analytic normalization, degree 2, level 1)
# Positive zeros: N+(T) = (T/(2π)) log(T/(2πe)) — wait, no.
# Riemann-von Mangoldt for degree d, conductor q L-function:
# N(T) = (d T/(2π)) log(T/(2π e)) + (T/(2π)) log(q) + O(log T)
# For d=2, q=1: N(T) = (T/π) log(T/(2πe)) (for total zeros, |Im|<T)
# For positive: N+(T) = (T/(2π)) log(T/(2π))^d ... hmm let me just fit.

n_idx_d = np.arange(1, len(ldelta)+1, dtype=float)

def smooth_template_d(T, a, b, c):
 # For degree-d (here d=2), heuristic: ~ (T/π)(log(T/a) - 1) + b
 return (T/PI) * (np.log(T/a) - 1) + b

from scipy.optimize import curve_fit
popt_d, _ = curve_fit(smooth_template_d, ldelta, n_idx_d, p0=[1.0, 0.0, 0.0], maxfev=5000)
print('ldelta fit:', popt_d)
unf_ldelta = smooth_template_d(ldelta, *popt_d)
sp_ldelta = np.diff(unf_ldelta)
print(f'ldelta spacings: mean={sp_ldelta.mean():.4f}, std={sp_ldelta.std():.4f}, n={len(sp_ldelta)}')
sp_ldelta /= sp_ldelta.mean()


ldelta: (540,) 9.222379399921103 499.42560626984044
ldelta fit: [6.27990287 3.17472688 0. ]
ldelta spacings: mean=0.9991, std=0.3439, n=539


/tmp/ipykernel_194/277136630.py:21: OptimizeWarning: Covariance of the parameters could not be estimated
 popt_d, _ = curve_fit(smooth_template_d, ldelta, n_idx_d, p0=[1.0, 0.0, 0.0], maxfev=5000)


In [14]:

# L(Δ,s) gives 539 spacings. Note: a fit returned 6.28 ≈ 2π — looks right.
# Now we have 4 functions: zeta (4999 spacings), ldh (4999), lchi (4999), ldelta (539).
# For matched comparison, subsample all to N=500 spacings.
# For ζ_δ (deformation control): The spec says it's stored as a list with shifted REAL parts,
# but since the spacings analysis uses only imaginary parts, ζ_δ ≡ ζ for this pipeline.
# We'll create ζ_δ_strong by perturbing IMAGINARY parts of ζ zeros — this is what would meaningfully
# test the spacings observable. Using δ=0.1 (10⁻¹) and shifting all zeros:
# That's what "strong" deformation would correspond to. But this is reconstructive — we'll document.

# Actually let's stick to the 4 functions we have authoritatively. For the missing 5th (ζ_δ_strong),
# we note: if spec is real-part shift only, it's IDENTICAL to ζ on this pipeline (zero separation between).
# We'll include ζ_δ AS ζ for the test, showing trivial p=1 by construction; OR omit it.
# We'll OMIT ζ_δ and report a 4x4 matrix, documenting why.

# Setup matched-size subsamples
import time
N_sub = 500
np.random.seed(42)

# Take first N_sub consecutive spacings (preserve serial structure for sliding window)
sp_zeta_s = sp_zeta[:N_sub].copy()
sp_ldh_s = sp_ldh[:N_sub].copy()
sp_lchi_s = sp_lchi[:N_sub].copy()
sp_ldelta_s = sp_ldelta[:N_sub].copy()

functions = {'ζ': sp_zeta_s, 'L_DH': sp_ldh_s, 'L(χ₅)': sp_lchi_s, 'L(Δ,s)': sp_ldelta_s}
print('Spacings counts:', {k: len(v) for k,v in functions.items()})


Spacings counts: {'ζ': 500, 'L_DH': 500, 'L(χ₅)': 500, 'L(Δ,s)': 500}


In [15]:

from ripser import ripser

def total_h1(spacings, dim=3, stride=1):
 pc = sliding_window(spacings, dim, stride)
 res = ripser(pc, maxdim=1)
 h1 = res['dgms'][1]
 if len(h1) == 0:
 return 0.0
 # Remove infinite features (shouldn't be H1 anyway)
 finite = h1[np.isfinite(h1[:,1])]
 return float((finite[:,1] - finite[:,0]).sum())

# Observed Total H1 per function
import time
obs_h1 = {}
for name, sp in functions.items():
 t0 = time.time()
 obs_h1[name] = total_h1(sp)
 print(f'{name}: TH1 = {obs_h1[name]:.4f}, time {time.time()-t0:.2f}s')


ζ: TH1 = 6.5846, time 0.19s
L_DH: TH1 = 4.6247, time 0.21s


L(χ₅): TH1 = 6.8096, time 0.20s
L(Δ,s): TH1 = 5.9590, time 0.20s


In [16]:

# Observed: ζ=6.58, L_DH=4.62, L(χ)=6.81, L(Δ)=5.96.
# L_DH has lowest TH1 — interesting. Now permutation test.

import itertools
B = 300 # number of permutations
np.random.seed(0)

names = list(functions.keys())
keys = list(functions.keys())
pairs = list(itertools.combinations(range(len(names)), 2))
print('Pairs:', [(names[i], names[j]) for i,j in pairs])

t_start = time.time()

p_matrix = np.ones((len(names), len(names)))
obs_stat = {}

for (i, j) in pairs:
 nA, nB = names[i], names[j]
 A = functions[nA]
 Bv = functions[nB]
 # Observed |TH1(A) - TH1(B)|
 T_obs = abs(obs_h1[nA] - obs_h1[nB])
 obs_stat[(nA, nB)] = T_obs
 
 pool = np.concatenate([A, Bv])
 nA_len = len(A)
 
 count_ge = 0
 for b in range(B):
 perm = np.random.permutation(len(pool))
 Ap = pool[perm[:nA_len]]
 Bp = pool[perm[nA_len:]]
 T_perm = abs(total_h1(Ap) - total_h1(Bp))
 if T_perm >= T_obs:
 count_ge += 1
 
 # p-value: (count_ge + 1) / (B + 1)
 p = (count_ge + 1) / (B + 1)
 p_matrix[i, j] = p
 p_matrix[j, i] = p
 elapsed = time.time() - t_start
 print(f'{nA} vs {nB}: T_obs={T_obs:.4f}, p={p:.4f}, count={count_ge}/{B}, elapsed={elapsed:.1f}s')

print(f'\nTotal time: {time.time()-t_start:.1f}s')


TimeoutError: Code execution timed out after 132.0 seconds